# Reading the evidence: a statistics toolkit (notebook)

Companion to `toolkit.md`. Every number on the page is computed here, card by card. Data: the monthly and daily
ETF closes of Lecture 5, fixed at the end of September 2026; simulations use fixed seeds.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats
from IPython.display import display

from trading_models.data.ohlc import load_ohlc
from trading_models.plotting import MUTED, SERIES, apply_theme, save_figure

apply_theme()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
FIG = ROOT / "lectures" / "statistics-toolkit" / "figures"
END = "2026-09-30"
TICKERS = ["SPY", "QQQ", "IWM", "EFA", "EEM", "XLE", "XLF", "XLK", "XLU", "VNQ", "AGG", "TLT", "GLD", "SLV", "GDX", "USO"]
Cl = load_ohlc(TICKERS, start="1993-01-01", cache=RAW / "nb05_ohlc.csv")["close"][TICKERS].loc[:END]
R = np.log(Cl).resample("ME").last().diff()   # monthly log returns

## 1. Standard error and t-statistic: SPY's drift (card 4)

In [2]:
r = R["SPY"].dropna()
n, m, sd = len(r), r.mean(), r.std()
se = sd / np.sqrt(n)
print(f"months {n}; mean {100*m:.3f}% a month; one month's sigma {100*sd:.2f}%")
print(f"standard error of the mean {100*se:.3f}% a month; t = {m/se:.2f}")
print(f"the mean in units of one month's sigma: {m/sd:.2f}")
print(f"annual: drift {1200*m:.2f}%, standard error {100*sd*np.sqrt(12)/np.sqrt(n/12):.2f}%")
# the same with only the last 5 years
r5 = r.loc["2021-10":]
print(f"last 5 years: mean {100*r5.mean():.3f}% a month, t = {r5.mean()/(r5.std()/np.sqrt(len(r5))):.2f} ({len(r5)} months)")

months 404; mean 0.856% a month; one month's sigma 4.27%
standard error of the mean 0.213% a month; t = 4.02
the mean in units of one month's sigma: 0.20
annual: drift 10.27%, standard error 2.55%
last 5 years: mean 1.070% a month, t = 1.84 (60 months)


## 2. What 2, 3 and 4 sigmas mean (two-sided chance under no effect) (card 5)

In [3]:
for k in (1, 2, 3, 4, 5):
    p = 2 * stats.norm.sf(k)
    print(f"{k} sigma: {100*p:.4f}%  (1 in {1/p:,.0f})")

1 sigma: 31.7311%  (1 in 3)
2 sigma: 4.5500%  (1 in 22)
3 sigma: 0.2700%  (1 in 370)
4 sigma: 0.0063%  (1 in 15,787)
5 sigma: 0.0001%  (1 in 1,744,278)


## 3. The Sharpe ratio's standard error: years needed for t = 2 (card 6)

In [4]:
for sr in (0.3, 0.5, 0.7, 1.0, 2.0):
    print(f"Sharpe {sr}: t after 10 years {sr*np.sqrt(10):.2f}; years for t = 2: {(2/sr)**2:.0f}")
# fat tails and skew widen it: Lo (2002)/Mertens style adjustment, computed on SPY monthly returns
from scipy.stats import skew, kurtosis
x = R["SPY"].dropna()
sr_m = x.mean() / x.std(); g3 = skew(x); g4 = kurtosis(x, fisher=False)
var_iid = (1 + 0.5 * sr_m**2) / len(x)
var_adj = (1 - g3 * sr_m + (g4 - 1) / 4 * sr_m**2) / len(x)
print(f"SPY monthly Sharpe {sr_m:.3f}, skew {g3:.2f}, kurtosis {g4:.2f}; se normal {np.sqrt(var_iid):.4f}, with skew and tails {np.sqrt(var_adj):.4f}")

Sharpe 0.3: t after 10 years 0.95; years for t = 2: 44
Sharpe 0.5: t after 10 years 1.58; years for t = 2: 16
Sharpe 0.7: t after 10 years 2.21; years for t = 2: 8
Sharpe 1.0: t after 10 years 3.16; years for t = 2: 4
Sharpe 2.0: t after 10 years 6.32; years for t = 2: 1
SPY monthly Sharpe 0.200, skew -0.73, kurtosis 4.36; se normal 0.0502, with skew and tails 0.0540


## 3b. Annualising: the square-root-of-time rule (card 3)

Volatility measured on daily, monthly and yearly returns, each scaled to a year. They agree only if returns are independent.

In [5]:
Dd = load_ohlc(TICKERS, start="1993-01-01", cache=RAW / "nb05_ohlc.csv")["close"][TICKERS].loc[:END]
rows = {}
for t in ["SPY", "TLT", "GLD", "USO"]:
    p = np.log(Dd[t].dropna())
    d = p.diff().dropna(); m = p.resample("ME").last().diff().dropna(); y = p.resample("YE").last().diff().dropna()
    rows[t] = {"daily x sqrt(252)": 100 * d.std() * np.sqrt(252), "monthly x sqrt(12)": 100 * m.std() * np.sqrt(12), "yearly": 100 * y.std()}
display(pd.DataFrame(rows).T.round(1))
# a trending (positively autocorrelated) series: same daily vol, more yearly vol
rng = np.random.default_rng(15)
e = rng.standard_normal(252 * 400); x = np.empty_like(e); x[0] = e[0]
for k in range(1, len(e)):
    x[k] = 0.1 * x[k - 1] + e[k]                       # autocorrelation 0.1 a day
yr = x.reshape(400, 252).sum(axis=1)
print(f"daily autocorrelation 0.1: daily vol x sqrt(252) = {x.std()*np.sqrt(252):.1f}, true yearly vol = {yr.std():.1f} (ratio {yr.std()/(x.std()*np.sqrt(252)):.2f})")

,daily x sqrt(252),monthly x sqrt(12),yearly
SPY,18.5,14.8,17.4
TLT,14.3,13.4,15.3
GLD,18.3,17.2,16.8
USO,37.6,40.4,44.8


daily autocorrelation 0.1: daily vol x sqrt(252) = 15.9, true yearly vol = 17.3 (ratio 1.08)


## 3c. Log returns, simple returns and the median outcome (card 2)

The average log return is the growth rate of the median path; the average simple return is that of the mean path. The gap is about half the variance (volatility drag).

In [6]:
s = R["SPY"].dropna(); simple = np.exp(s) - 1
print(f"SPY monthly, annualised: average simple return {1200*simple.mean():.2f}%, average log return {1200*s.mean():.2f}%, "
      f"half the variance {100*0.5*12*s.var():.2f}%")
print("+50% then -50%: average simple return 0%, average log return", f"{100*np.mean(np.log([1.5, 0.5])):.1f}%, wealth {1.5*0.5:.2f}")
# 20,000 paths of 20 years: 7% average log return a year, 20% volatility
rng = np.random.default_rng(16)
lr = rng.normal(0.07 / 12, 0.20 / np.sqrt(12), (20000, 240)).sum(axis=1)
W = np.exp(lr)
print(f"20 years, average log return 7% a year, volatility 20%: median wealth {np.median(W):.2f} (exp(20 x 0.07) = {np.exp(1.4):.2f}); "
      f"mean wealth {W.mean():.2f}; share of paths below the mean {(W < W.mean()).mean():.0%}")

SPY monthly, annualised: average simple return 11.40%, average log return 10.27%, half the variance 1.10%
+50% then -50%: average simple return 0%, average log return -14.4%, wealth 0.75


20 years, average log return 7% a year, volatility 20%: median wealth 4.08 (exp(20 x 0.07) = 4.06); mean wealth 6.05; share of paths below the mean 67%


### 3d. The diversification return: adding log returns across assets (card 1)

Equal weights on the 16 ETFs, rebalanced monthly, from mid-2006 (all funds available).

In [7]:
m = Cl.loc["2006-07":].resample("ME").last()
simple, logr = m.pct_change().dropna(), np.log(m).diff().dropna()
w = 1 / len(TICKERS)
port = np.log1p((simple * w).sum(axis=1))          # true log return of the rebalanced portfolio
naive = (logr * w).sum(axis=1)                      # weighted average of the assets' log returns
avg_var, p_var = 12 * (logr.var() * w).sum(), 12 * port.var()
print(f"portfolio log return {1200*port.mean():.2f}% a year; weighted log returns {1200*naive.mean():.2f}%; gap {1200*(port-naive).mean():.2f}")
print(f"half (average variance - portfolio variance) {50*(avg_var-p_var):.2f}; average asset vol {100*np.sqrt(12*logr.var().mean()):.1f}%, portfolio vol {100*np.sqrt(p_var):.1f}%")

portfolio log return 8.43% a year; weighted log returns 6.70%; gap 1.72
half (average variance - portfolio variance) 1.76; average asset vol 23.3%, portfolio vol 13.8%


### 3e. The same bet at different sizes (card 2)

Sharpe ratio 0.5 held at several volatilities: growth g = σ(SR − σ/2), and a Sharpe ratio built on growth, g/σ. Checked by simulation at 20% volatility.

In [8]:
sr = 0.5
display(pd.DataFrame({f"{v:.0%}": {"Sharpe": sr, "growth % a year": 100 * v * (sr - v / 2), "Sharpe built on growth": sr - v / 2} for v in (0.1, 0.2, 0.5, 1.0)}).T.round(2))
rng = np.random.default_rng(17)
x = rng.normal(sr * 0.2 / 252, 0.2 / np.sqrt(252), 252 * 2000)
print(f"simulated, 20% volatility: arithmetic {100*252*x.mean():.2f}% a year, growth {100*252*np.log1p(x).mean():.2f}% a year")

,Sharpe,growth % a year,Sharpe built on growth
10%,0.5,4.5,0.45
20%,0.5,8.0,0.40
50%,0.5,12.5,0.25
100%,0.5,0.0,0.00


simulated, 20% volatility: arithmetic 9.84% a year, growth 7.83% a year


## 4. Many tries: the best of N zero-edge strategies (card 8)

In [9]:
rng = np.random.default_rng(1)
for N in (1, 10, 30, 100, 1000):
    best = rng.standard_normal((20000, N)).max(axis=1)
    print(f"best of {N:>4}: median t {np.median(best):.2f}; chance the best exceeds 2: {(best > 2).mean():.0%}")

best of    1: median t -0.01; chance the best exceeds 2: 2%


best of   10: median t 1.49; chance the best exceeds 2: 20%
best of   30: median t 1.99; chance the best exceeds 2: 50%


best of  100: median t 2.46; chance the best exceeds 2: 90%


best of 1000: median t 3.19; chance the best exceeds 2: 100%


In [10]:
x = np.linspace(-4.5, 5.5, 600)
rng = np.random.default_rng(1)
best1000 = rng.standard_normal((20000, 1000)).max(axis=1)
kde = stats.gaussian_kde(best1000)
fig = go.Figure()
fig.add_scatter(x=x, y=stats.norm.pdf(x), name="t of one zero-edge strategy", line=dict(color=SERIES[0], width=2))
fig.add_scatter(x=x, y=kde(x), name="t of the best of 1,000 zero-edge strategies", line=dict(color=SERIES[1], width=2))
for k in (2, 3, 4):
    fig.add_vline(x=k, line=dict(color=MUTED, dash="dot", width=1))
    lab = {2: "2 sigma: 4.6%", 3: "3 sigma: 0.27%", 4: "4 sigma: 0.006%"}[k]
    fig.add_annotation(x=k, y=1.0, yref="paper", text=lab, showarrow=False, xanchor="left", xshift=4, yshift={2: 0, 3: -22, 4: 0}[k])
fig.update_layout(title="The t-statistic of a strategy with no edge: one try, and the best of 1,000",
                  xaxis_title="t-statistic", yaxis_title="density", height=430, width=1000, margin=dict(t=80),
                  legend=dict(orientation="h", y=-0.2, x=0))
save_figure(fig, "T_01_sigmas_and_tries", FIG)
fig

## 5. Shuffle and bootstrap: the luck band for SPY's 12-month rule (card 9)

In [11]:
r = R["SPY"].dropna()
s = np.sign(r.rolling(12).sum()).shift(1)
d = pd.concat([s, r], axis=1, keys=["s", "r"]).dropna()
rule = 1200 * (d.s * d.r).mean()
rng = np.random.default_rng(5)
shuf = np.array([1200 * (rng.permutation(d.s.values) * d.r.values).mean() for _ in range(5000)])
print(f"rule {rule:.2f}% a year; shuffled mean {shuf.mean():.2f}; 5th-95th {np.percentile(shuf,5):.2f} to {np.percentile(shuf,95):.2f}; "
      f"share of shuffles at or above the rule {(shuf >= rule).mean():.1%}")
# bootstrap of the rule's mean return: resample months with replacement
sr = (d.s * d.r).values
boot = np.array([1200 * rng.choice(sr, len(sr)).mean() for _ in range(5000)])
print(f"bootstrap of the rule's return: 5th-95th {np.percentile(boot,5):.2f} to {np.percentile(boot,95):.2f}")

rule 10.63% a year; shuffled mean 6.55; 5th-95th 3.31 to 9.84; share of shuffles at or above the rule 2.4%


bootstrap of the rule's return: 5th-95th 6.48 to 14.85


In [12]:
fig = go.Figure()
fig.add_histogram(x=shuf, nbinsx=60, marker_color=MUTED, name="5,000 shuffled position paths")
fig.add_vline(x=rule, line=dict(color=SERIES[1], width=3))
fig.add_annotation(x=rule, y=1, yref="paper", text=f"the 12-month rule: {rule:.1f}%", showarrow=False, xanchor="right", xshift=-6, yshift=-10)
fig.add_vline(x=shuf.mean(), line=dict(color=SERIES[0], width=2, dash="dash"))
fig.add_annotation(x=shuf.mean(), y=0.92, yref="paper", text=f"shuffled average {shuf.mean():.1f}% = drift part", showarrow=False, xanchor="right", xshift=-6)
fig.update_layout(title="SPY: the trend rule against the same positions placed at random (% a year)",
                  xaxis_title="average return, % a year", yaxis_title="number of shuffled paths", height=430, width=1000,
                  margin=dict(t=80), showlegend=False)
save_figure(fig, "T_02_shuffle_band", FIG)
fig

## 6. Median, percentiles and rank correlation (card 10)

In [13]:
rows = {}
for t in TICKERS:
    r = R[t].dropna(); s = np.sign(r.rolling(12).sum()).shift(1); dd = pd.concat([s, r], axis=1, keys=["s", "r"]).dropna()
    tot = (dd.s * dd.r).mean(); rows[t] = 1200 * (tot - dd.s.mean() * dd.r.mean())
tim = pd.Series(rows)
print(f"timing part across 16 ETFs: mean {tim.mean():.2f}, median {tim.median():.2f}")
print(f"without USO: mean {tim.drop('USO').mean():.2f}, median {tim.drop('USO').median():.2f}")
# one outlier and correlation
rng = np.random.default_rng(3)
a = rng.standard_normal(30); b = rng.standard_normal(30)
print(f"two unrelated series, 30 points: Pearson {np.corrcoef(a, b)[0,1]:.2f}, Spearman {stats.spearmanr(a, b)[0]:.2f}")
a2, b2 = np.append(a, 8), np.append(b, 8)
print(f"add one joint extreme point: Pearson {np.corrcoef(a2, b2)[0,1]:.2f}, Spearman {stats.spearmanr(a2, b2)[0]:.2f}")

timing part across 16 ETFs: mean 2.55, median 2.18
without USO: mean 2.23, median 2.08
two unrelated series, 30 points: Pearson 0.01, Spearman 0.12
add one joint extreme point: Pearson 0.65, Spearman 0.20


## 7. Correlation and its noise (card 11)

In [14]:
for n in (12, 36, 60, 120, 400):
    print(f"n = {n}: standard error near zero about 1/sqrt(n) = {1/np.sqrt(n):.2f}")
x = R[["SPY", "TLT"]].dropna()
full = x.corr().iloc[0, 1]
roll = x.SPY.rolling(36).corr(x.TLT).dropna()
print(f"SPY-TLT monthly correlation {x.index[0]:%Y-%m} to {x.index[-1]:%Y-%m}: {full:.2f}; 36-month windows from {roll.min():.2f} ({roll.idxmin():%Y-%m}) to {roll.max():.2f} ({roll.idxmax():%Y-%m})")
# how wide is pure noise for 36 months?
rng = np.random.default_rng(7)
z = np.array([np.corrcoef(rng.standard_normal(36), rng.standard_normal(36))[0, 1] for _ in range(20000)])
print(f"two unrelated series, 36 months: 5th-95th percentile of the correlation {np.percentile(z,5):.2f} to {np.percentile(z,95):.2f}")
# levels, not returns: two independent random walks
lv = []
for _ in range(2000):
    p1 = np.cumsum(rng.standard_normal(240)); p2 = np.cumsum(rng.standard_normal(240))
    lv.append(np.corrcoef(p1, p2)[0, 1])
lv = np.abs(lv)
print(f"two independent random walks, 240 months, correlation of PRICE LEVELS: median |corr| {np.median(lv):.2f}; share above 0.5: {(lv > 0.5).mean():.0%}")

n = 12: standard error near zero about 1/sqrt(n) = 0.29
n = 36: standard error near zero about 1/sqrt(n) = 0.17
n = 60: standard error near zero about 1/sqrt(n) = 0.13
n = 120: standard error near zero about 1/sqrt(n) = 0.09
n = 400: standard error near zero about 1/sqrt(n) = 0.05
SPY-TLT monthly correlation 2002-08 to 2026-09: -0.09; 36-month windows from -0.79 (2013-01) to 0.68 (2024-12)


two unrelated series, 36 months: 5th-95th percentile of the correlation -0.28 to 0.28


two independent random walks, 240 months, correlation of PRICE LEVELS: median |corr| 0.40; share above 0.5: 38%


## 8. Maximum drawdown as a statistic: simulated strategy with Sharpe 0.5 and 10% volatility (card 13)

In [15]:
rng = np.random.default_rng(11)
mu, vol = 0.05 / 12, 0.10 / np.sqrt(12)
paths = rng.normal(mu, vol, (10000, 480))
rows = {}
for yrs in (5, 10, 20, 40):
    eq = np.cumsum(paths[:, :12 * yrs], axis=1)                     # log equity
    dd = eq - np.maximum.accumulate(np.concatenate([np.zeros((len(eq), 1)), eq], axis=1), axis=1)[:, 1:]
    mdd = 100 * (np.exp(dd.min(axis=1)) - 1)
    rows[yrs] = {"median": np.median(mdd), "5th pct (bad luck)": np.percentile(mdd, 5), "95th pct (good luck)": np.percentile(mdd, 95)}
mdd_tab = pd.DataFrame(rows).T
display(mdd_tab.round(1))

,median,5th pct (bad luck),95th pct (good luck)
5,-14.0,-26.8,-7.5
10,-18.2,-32.3,-10.6
20,-22.6,-37.6,-14.2
40,-26.9,-42.7,-18.2


In [16]:
fig = go.Figure()
yrs = [f"{y} years" for y in mdd_tab.index]
fig.add_bar(x=yrs, y=mdd_tab["95th pct (good luck)"], name="good luck (95th percentile)", marker_color=SERIES[0])
fig.add_bar(x=yrs, y=mdd_tab["median"], name="median", marker_color=MUTED)
fig.add_bar(x=yrs, y=mdd_tab["5th pct (bad luck)"], name="bad luck (5th percentile)", marker_color=SERIES[1])
fig.update_layout(barmode="group", title="Worst drawdown of the same strategy (Sharpe 0.5, 10% volatility) over longer histories",
                  xaxis_title="length of history", xaxis_type="category", yaxis_title="worst drawdown, %", height=430, width=1000, margin=dict(t=80),
                  legend=dict(orientation="h", y=-0.2, x=0))
save_figure(fig, "T_03_drawdown_by_length", FIG)
fig

### 8b. Expected worst drawdown from the Sharpe ratio (in units of annual volatility) (card 13)

Normal, independent monthly returns; median of 10,000 paths. Multiply by the strategy's volatility: at 10% volatility, 2.0 means a 20% drawdown in log terms.

In [17]:
rng = np.random.default_rng(12)
z = rng.standard_normal((10000, 480))
rows = {}
for sr in (0.3, 0.5, 0.7, 1.0):
    lp = sr / 12 + z / np.sqrt(12)              # monthly log return with annual vol 1 and Sharpe sr
    row = {}
    for yrs in (5, 10, 20, 40):
        eq = np.concatenate([np.zeros((len(lp), 1)), np.cumsum(lp[:, :12 * yrs], axis=1)], axis=1)
        mdd = (eq - np.maximum.accumulate(eq, axis=1)).min(axis=1)
        row[f"{yrs} years"] = -np.median(mdd)
    rows[f"Sharpe {sr}"] = row
dd_sr = pd.DataFrame(rows).T
display(dd_sr.round(2))

,5 years,10 years,20 years,40 years
Sharpe 0.3,1.75,2.39,3.13,4.03
Sharpe 0.5,1.52,2.00,2.54,3.15
Sharpe 0.7,1.33,1.73,2.14,2.60
Sharpe 1.0,1.12,1.42,1.72,2.05


### 8c. A clean backtest: will the drawdown come? (card 13)

Sharpe 0.5, 10% volatility. The backtest's 10 years happened to show a worst drawdown of only 10% (good luck). How likely is a deeper one in the next 10 and 20 years of the same strategy?

In [18]:
rng = np.random.default_rng(13)
lp = rng.normal(0.05 / 12, 0.10 / np.sqrt(12), (20000, 240))
for yrs in (10, 20):
    eq = np.concatenate([np.zeros((len(lp), 1)), np.cumsum(lp[:, :12 * yrs], axis=1)], axis=1)
    mdd = 100 * (np.exp((eq - np.maximum.accumulate(eq, axis=1)).min(axis=1)) - 1)
    print(f"next {yrs} years: chance of a drawdown deeper than -10%: {(mdd < -10).mean():.0%}; deeper than -20%: {(mdd < -20).mean():.0%}")

next 10 years: chance of a drawdown deeper than -10%: 96%; deeper than -20%: 39%


next 20 years: chance of a drawdown deeper than -10%: 100%; deeper than -20%: 65%


## 8d. R²: how much a forecast explains, and what that is worth (card 12)

A forecast x with monthly R² against next month's return; the strategy holds a position proportional to x.

In [19]:
rng = np.random.default_rng(14)
T = 1_000_000
for r2 in (0.001, 0.005, 0.01, 0.02, 0.05):
    rho = np.sqrt(r2)
    x = rng.standard_normal(T); r = rho * x + np.sqrt(1 - r2) * rng.standard_normal(T)
    pnl = x * r
    print(f"R² {r2:.1%}: correlation {rho:.3f}; annual Sharpe of the position x: {np.sqrt(12) * pnl.mean() / pnl.std():.2f} (rule of thumb sqrt(12 R²) = {np.sqrt(12 * r2):.2f})")

R² 0.1%: correlation 0.032; annual Sharpe of the position x: 0.11 (rule of thumb sqrt(12 R²) = 0.11)


R² 0.5%: correlation 0.071; annual Sharpe of the position x: 0.25 (rule of thumb sqrt(12 R²) = 0.24)


R² 1.0%: correlation 0.100; annual Sharpe of the position x: 0.35 (rule of thumb sqrt(12 R²) = 0.35)


R² 2.0%: correlation 0.141; annual Sharpe of the position x: 0.49 (rule of thumb sqrt(12 R²) = 0.49)


R² 5.0%: correlation 0.224; annual Sharpe of the position x: 0.76 (rule of thumb sqrt(12 R²) = 0.77)


## 9. When the sigma is wrong: overlapping 12-month returns (card 7)

In [20]:
r = R["SPY"].dropna()
y12 = r.rolling(12).sum().dropna()            # 12-month returns, one per month: they overlap by 11 months
naive_se = y12.std() / np.sqrt(len(y12))
honest = r.rolling(12).sum().iloc[11::12].dropna()   # non-overlapping years
honest_se = honest.std() / np.sqrt(len(honest))
print(f"mean 12-month return {100*y12.mean():.2f}%")
print(f"naive t with {len(y12)} overlapping windows: {y12.mean()/naive_se:.2f}")
print(f"t with {len(honest)} non-overlapping years: {honest.mean()/honest_se:.2f}")
print(f"ratio of naive to honest t: {(y12.mean()/naive_se)/(honest.mean()/honest_se):.2f} (about sqrt(12) = {np.sqrt(12):.2f})")

mean 12-month return 10.20%
naive t with 393 overlapping windows: 12.56
t with 33 non-overlapping years: 3.41
ratio of naive to honest t: 3.69 (about sqrt(12) = 3.46)
